# `klygo.files` (`download.py`): Remote and local file downloads

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/IchigoMazone/klygo/blob/v2-release/tutorials/files/Document_Files_Download.ipynb)


Bring a single local file or URL into a destination directory, with explicit overwrite and checksum options.

Run this notebook from top to bottom. It creates only temporary example files.

In [ ]:
!pip install git+https://github.com/IchigoMazone/klygo.git@v2-release

## Prepare sample files

Run this setup again to reset the examples. No Drive mount or manual upload is required.

In [1]:
from tempfile import TemporaryDirectory
from klygo import files

# Run this cell again to replace only this notebook's temporary workspace.
if "tutorial_workspace" in globals():
    tutorial_workspace.cleanup()
tutorial_workspace = TemporaryDirectory(prefix="klygo-files-")
root = files.path(tutorial_workspace.name)
print("Example directory:", root)

Example directory: C:\Users\Mine\AppData\Local\Temp\klygo-files-fg8x0fj5


In [2]:
print("A local source makes the core example independent of network access.")

A local source makes the core example independent of network access.


## Jump to an API

[`download`](#download)

<a id="download"></a>

### `files.download`

Download a URL or copy a local file while preserving its name.

**Signature:** `files.download(source: 'PathInput', output_dir: 'PathInput' = '.', overwrite: 'bool' = False, verbose: 'bool' = True, *, timeout: 'float' = 30.0, retries: 'int' = 0, expected_sha256: 'str | None' = None) -> 'Path'`

- `source`: Source URL or filesystem path.
- `output_dir`: Directory in which the output file is created.
- `overwrite`: Allow an existing destination to be replaced.
- `verbose`: Display a progress indicator.
- `timeout`: Network timeout in seconds.
- `retries`: Additional attempts after a network or incomplete-transfer error.
- `expected_sha256`: Expected SHA-256 digest, checked before the file is published.

**Returns:** `pathlib.Path` — Path to the downloaded or copied file.

Local files and HTTP/HTTPS/FTP URLs are supported. For URLs, `timeout` is per network attempt and `retries` adds attempts for network or incomplete-transfer errors. `expected_sha256` checks the file before publication. In Google Colab, a local source triggers the browser download helper and returns the source path; it does not copy into `output_dir`.

In [3]:
source = root / 'sample.bin'
source.write_bytes(b'example weights\n')
checksum = files.hash(source)
received = files.download(source, output_dir=root / 'downloads', verbose=False, expected_sha256=checksum)
print(received, files.compare(source, received))
try:
    files.download(source, output_dir=root / 'downloads', verbose=False)
except FileExistsError:
    print('Destination exists; set overwrite=True only when replacement is intended.')

C:\Users\Mine\AppData\Local\Temp\klygo-files-fg8x0fj5\downloads\sample.bin True
Destination exists; set overwrite=True only when replacement is intended.


**Another useful pattern**

In [4]:
# An incorrect digest is rejected before replacing the destination.
try:
    files.download(source, root / 'verified', verbose=False, expected_sha256='0' * 64)
except ValueError as error:
    print(type(error).__name__, '-', error)
print('Verified directory entries:', files.list_entries(root / 'verified'))

ValueError - Source file does not match expected_sha256
Verified directory entries: []


**Expected error and recovery**

In [5]:
try:
    files.download(source, root / "invalid", timeout=0, verbose=False)
except ValueError as error:
    print("ValueError:", error)

try:
    files.download(source, root / "invalid", retries=-1, verbose=False)
except ValueError as error:
    print("ValueError:", error)

ValueError: timeout must be a positive number of seconds
ValueError: retries must be a non-negative integer


## Put it together

In [6]:
local_asset = root / 'final.bin'
local_asset.write_bytes(b'final model bytes')
print('Check before transfer:', files.hash(local_asset))
print('Received:', files.download(local_asset, root / 'artifacts', verbose=False, expected_sha256=files.hash(local_asset)))

Check before transfer: ed25de6efaf0c30da3b1cca1b2943e076b7953a32192ede2211e8214e78a1e5a
Received: C:\Users\Mine\AppData\Local\Temp\klygo-files-fg8x0fj5\artifacts\final.bin


## Remember

- Local-source `download()` is network-free locally (and triggers a browser download in Colab); URL downloads verify `expected_sha256` before publishing the file.
- Next in `klygo.files`: start over at [Document_Files_Paths.ipynb](Document_Files_Paths.ipynb) ([Colab](https://colab.research.google.com/github/IchigoMazone/klygo/blob/v2-release/tutorials/files/Document_Files_Paths.ipynb)) or return to [Document_Files_Overview.ipynb](Document_Files_Overview.ipynb) ([Colab](https://colab.research.google.com/github/IchigoMazone/klygo/blob/v2-release/tutorials/files/Document_Files_Overview.ipynb)).
- Related Klygo modules: [`klygo.archive`](../archive/Document_Archive_Overview.ipynb) · [`klygo.config`](../config/Document_Config_Overview.ipynb).

## URL example

To fetch a public URL, replace the example URL below with a real file endpoint. A network connection is required. Set `expected_sha256` when the publisher supplies a digest.

```python
url = "https://example.org/files/model.bin"  # Replace with a real download URL.
model = files.download(url, output_dir=root / "models", timeout=20, retries=2)
print(model)
```